# ISL Landmark Extraction (Kaggle)

Runs INCLUDE video -> MediaPipe Holistic -> `.npy` landmarks in the cloud so your local CPU stays free. **Train locally afterwards** on your GPU.

## Workflow
1. Run the **Setup** cell (installs pinned deps).
2. Run the two **Write source** cells.
3. Run **Locate dataset** (mount the INCLUDE dataset via right-side `+ Add data` -> Kaggle Datasets -> `daskoushik/include`).
4. **Smoke test** 5 videos.
5. **Full extraction** (about 1-4 h on Kaggle CPU; resumable).
6. **Zip** the landmarks, then download `landmarks.zip` from the output panel.

Unzip locally into `data/landmarks/`, then run `utils/make_splits.py --top_n 50` and `training/train.py` on your GPU.


In [ ]:
!pip install -q mediapipe==0.10.21 opencv-contrib-python numpy tqdm

In [ ]:
import pathlib
_p = pathlib.Path('models/landmark_extractor.py')
_p.parent.mkdir(parents=True, exist_ok=True)
_p.write_text(r'''"""Shared MediaPipe Holistic landmark extractor.

This module is used by BOTH `preprocessing/extract_landmarks.py` (offline,
over the INCLUDE dataset) and `inference/live_predict.py` (live webcam), so
that training and inference see exactly the same feature representation.

Per-frame feature layout (float32):

    [ pose(33*3) | left_hand(21*3) | right_hand(21*3) | face(468*3) | flags(3) ]

    flags = [pose_detected, left_hand_detected, right_hand_detected]

With include_face=True the feature dim is 99 + 126 + 1404 + 3 = 1632.
With include_face=False it is 99 + 126 + 3 = 234.

Normalization: all coordinates are re-expressed relative to the mid-shoulder
point and divided by the shoulder width, making the representation
translation- and scale-invariant. When the pose is missing in a frame, the
previous frame's origin/scale are reused (state is reset at the start of
every clip).
"""

from __future__ import annotations

import contextlib
from typing import Optional

import cv2
import numpy as np

POSE_LM = 33
HAND_LM = 21
FACE_LM = 468

POSE_DIM = POSE_LM * 3          # 99
HAND_DIM = HAND_LM * 3          # 63  (per hand)
HANDS_DIM = 2 * HAND_DIM        # 126
FACE_DIM = FACE_LM * 3          # 1404
FLAGS_DIM = 3                   # pose, left hand, right hand


def feature_dim(include_face: bool) -> int:
    """Total per-frame feature dimensionality."""
    return POSE_DIM + HANDS_DIM + (FACE_DIM if include_face else 0) + FLAGS_DIM


def segment_offsets(include_face: bool) -> dict:
    """Start offsets of each feature segment within a frame vector."""
    off = {"pose": 0, "left_hand": POSE_DIM, "right_hand": POSE_DIM + HAND_DIM}
    face_start = POSE_DIM + HANDS_DIM
    flags_start = face_start + (FACE_DIM if include_face else 0)
    if include_face:
        off["face"] = face_start
    off["flags"] = flags_start
    return off


def _to_array(landmarks, n: int) -> np.ndarray:
    """MediaPipe landmark list -> (n, 3) float32 array, zeros if absent."""
    out = np.zeros((n, 3), dtype=np.float32)
    if landmarks is not None:
        for i, lm in enumerate(landmarks.landmark[:n]):
            out[i, 0] = lm.x
            out[i, 1] = lm.y
            out[i, 2] = lm.z
    return out


class LandmarkExtractor:
    """Stateful MediaPipe Holistic wrapper producing normalized feature frames.

    Use as a context manager::

        with LandmarkExtractor(include_face=True) as ext:
            vec = ext.process_frame(bgr_frame)   # -> (D,) float32
            seq = ext.extract_video("clip.mov")  # -> (T, D) float32
    """

    def __init__(
        self,
        include_face: bool = True,
        model_complexity: int = 1,
        static_image_mode: bool = False,
        min_detection_confidence: float = 0.5,
        min_tracking_confidence: float = 0.5,
    ):
        import mediapipe as mp  # deferred so importing this module is cheap

        self.include_face = include_face
        self._mp = mp
        self._holistic = mp.solutions.holistic.Holistic(
            static_image_mode=static_image_mode,
            model_complexity=model_complexity,
            min_detection_confidence=min_detection_confidence,
            min_tracking_confidence=min_tracking_confidence,
        )
        # Normalization state, reset per clip.
        self._origin = np.zeros(3, dtype=np.float32)
        self._scale = 1.0

    # -- context manager ----------------------------------------------------
    def __enter__(self):
        return self

    def __exit__(self, *exc):
        self.close()
        return False

    def close(self):
        self._holistic.close()

    def reset(self):
        """Reset normalization calibration (call at the start of a clip)."""
        self._origin = np.zeros(3, dtype=np.float32)
        self._scale = 1.0

    # -- core ----------------------------------------------------------------
    def process_frame(self, frame_bgr: np.ndarray) -> np.ndarray:
        """Run Holistic on one BGR frame -> (D,) float32 feature vector."""
        rgb = cv2.cvtColor(frame_bgr, cv2.COLOR_BGR2RGB)
        res = self._holistic.process(rgb)

        pose = _to_array(res.pose_landmarks, POSE_LM)
        left = _to_array(res.left_hand_landmarks, HAND_LM)
        right = _to_array(res.right_hand_landmarks, HAND_LM)
        face = _to_array(res.face_landmarks, FACE_LM) if self.include_face else None

        pose_ok = res.pose_landmarks is not None
        left_ok = res.left_hand_landmarks is not None
        right_ok = res.right_hand_landmarks is not None

        # --- normalization calibration -------------------------------------
        if pose_ok:
            ls, rs = pose[11], pose[12]  # left / right shoulder
            width = float(np.linalg.norm(ls - rs))
            if width > 1e-3:
                self._origin = (ls + rs) / 2.0
                self._scale = width

        def norm(arr: np.ndarray) -> np.ndarray:
            return ((arr - self._origin) / self._scale).astype(np.float32)

        parts = [
            norm(pose).reshape(-1),
            norm(left).reshape(-1),
            norm(right).reshape(-1),
        ]
        if face is not None:
            parts.append(norm(face).reshape(-1))
        parts.append(np.array([pose_ok, left_ok, right_ok], dtype=np.float32))
        return np.concatenate(parts, axis=0)

    def extract_video(self, video_path: str, max_frames: Optional[int] = 200) -> np.ndarray:
        """Decode a video file -> (T, D) float32 landmark sequence."""
        self.reset()
        cap = cv2.VideoCapture(video_path)
        if not cap.isOpened():
            raise IOError(f"Cannot open video: {video_path}")
        feats = []
        try:
            while True:
                if max_frames is not None and len(feats) >= max_frames:
                    break
                ok, frame = cap.read()
                if not ok:
                    break
                feats.append(self.process_frame(frame))
        finally:
            cap.release()
        if not feats:
            return np.zeros((0, feature_dim(self.include_face)), dtype=np.float32)
        return np.stack(feats, axis=0)
''', encoding='utf-8')
print('wrote', _p, _p.stat().st_size, 'bytes')


In [ ]:
import pathlib
_p = pathlib.Path('preprocessing/extract_landmarks.py')
_p.parent.mkdir(parents=True, exist_ok=True)
_p.write_text(r'''"""Extract landmark sequences from all INCLUDE videos into .npy files.

Walks raw_data/<Category_Part>/<Category>/<NN. Word>/<video>.MOV and writes
data/landmarks/<Category>/<NN. Word>/<video>.npy with shape (T, D) float32.

Resumable: existing .npy files are skipped, so the job can be stopped and
restarted (run overnight).

Usage:
    python preprocessing/extract_landmarks.py --raw_dir raw_data --out_dir data/landmarks
    python preprocessing/extract_landmarks.py --limit_videos 5          # quick test
    python preprocessing/extract_landmarks.py --only_category Greetings --no_face
"""

from __future__ import annotations

import argparse
import sys
import time
from pathlib import Path

import numpy as np
from tqdm import tqdm

sys.path.insert(0, str(Path(__file__).resolve().parents[1]))
from models.landmark_extractor import LandmarkExtractor, feature_dim  # noqa: E402

VIDEO_EXTS = {".MOV", ".MP4", ".AVI", ".MKV", ".MOV"}


def find_videos(raw_dir: Path, only_category: str | None):
    """Yield (video_path, word_dir) pairs for every video under raw_dir."""
    for video in sorted(raw_dir.rglob("*")):
        if not video.is_file() or video.suffix.upper() not in VIDEO_EXTS:
            continue
        word_dir = video.parent  # e.g. .../Greetings/48. Hello
        if only_category and only_category.lower() not in str(word_dir).lower():
            continue
        yield video, word_dir


def resolve_raw_dir(raw_dir: Path) -> Path:
    """Return the directory that actually holds the INCLUDE category folders.

    Handles both layouts:
      data/raw_data/<CategoryPart>/<Category>/<NN. Word>/<video>
      data/raw_data/raw_data/<CategoryPart>/<Category>/<NN. Word>/<video>
    (the nested `raw_data` arises when the INCLUDE zip is unpacked inside
    data/raw_data).
    """
    nested = raw_dir / "raw_data"
    if nested.is_dir() and not any(p.is_file() for p in raw_dir.iterdir()):
        return nested
    return raw_dir


def main() -> None:
    ap = argparse.ArgumentParser(description=__doc__)
    ap.add_argument("--raw_dir", default="data/raw_data", type=Path)
    ap.add_argument("--out_dir", default="data/landmarks", type=Path)
    ap.add_argument("--include_face", dest="include_face", action="store_true", default=True)
    ap.add_argument("--no_face", dest="include_face", action="store_false",
                    help="Exclude 468 facial landmarks (D=234 instead of 1632)")
    ap.add_argument("--model_complexity", type=int, default=1, choices=[0, 1, 2])
    ap.add_argument("--max_frames", type=int, default=200,
                    help="Cap frames per video (INCLUDE signs are short)")
    ap.add_argument("--only_category", type=str, default=None,
                    help="Substring filter on category, e.g. 'Greetings'")
    ap.add_argument("--limit_videos", type=int, default=None,
                    help="Process only the first N videos (smoke test)")
    args = ap.parse_args()
    args.raw_dir = resolve_raw_dir(args.raw_dir)

    jobs = list(find_videos(args.raw_dir, args.only_category))
    if args.limit_videos:
        jobs = jobs[: args.limit_videos]
    if not jobs:
        print(f"No videos found under {args.raw_dir}")
        return

    print(f"{len(jobs)} videos | feature dim D = {feature_dim(args.include_face)}")
    extractor = LandmarkExtractor(
        include_face=args.include_face, model_complexity=args.model_complexity
    )

    n_done = n_skip = n_fail = 0
    t0 = time.time()
    with extractor:
        for video, word_dir in tqdm(jobs, desc="extracting", unit="video"):
            out_path = args.out_dir / word_dir.relative_to(args.raw_dir) / (
                video.stem + ".npy"
            )
            if out_path.exists():  # resumable
                n_skip += 1
                continue
            try:
                seq = extractor.extract_video(str(video), max_frames=args.max_frames)
            except Exception as exc:  # noqa: BLE001
                print(f"\nFAILED {video}: {exc}")
                n_fail += 1
                continue
            if seq.shape[0] == 0:
                print(f"\nEMPTY {video}")
                n_fail += 1
                continue
            out_path.parent.mkdir(parents=True, exist_ok=True)
            np.save(out_path, seq.astype(np.float32))
            n_done += 1

    dt = time.time() - t0
    print(f"\nDone in {dt/60:.1f} min | extracted {n_done} | skipped {n_skip} "
          f"| failed {n_fail} | out: {args.out_dir}")


if __name__ == "__main__":
    main()
''', encoding='utf-8')
print('wrote', _p, _p.stat().st_size, 'bytes')


In [ ]:
import sys, pathlib
sys.path.insert(0, '.')
from preprocessing.extract_landmarks import find_videos, resolve_raw_dir

def locate_raw_dir(base=pathlib.Path('/kaggle/input'), exts=('.MOV','.MP4','.AVI','.MKV')):
    hits = [p for p in base.rglob('*') if p.is_file() and p.suffix.upper() in exts]
    if not hits:
        raise SystemExit('No videos under ' + str(base) + ' - mount INCLUDE dataset')
    parents = [set(p.parents) for p in hits]
    common = set.intersection(*parents)
    return max(common, key=lambda d: len(str(d))), len(hits)

raw_dir, n = locate_raw_dir()
raw_dir = resolve_raw_dir(raw_dir)
print('raw_dir ->', raw_dir)
jobs = list(find_videos(raw_dir, None))
print('videos:', len(jobs), '| unique word folders:', len({str(p.parent) for p,_ in jobs}))
_RAW_DIR = str(pathlib.Path('/kaggle/input'))

# 4) Smoke test (5 videos)

Quick sanity check of the extraction and saved `.npy` format before the long run.

In [ ]:
import subprocess, sys
r = subprocess.run([sys.executable, 'preprocessing/extract_landmarks.py',
                    '--raw_dir', _RAW_DIR, '--out_dir', 'data/landmarks',
                    '--limit_videos', '5'])
print('exit', r.returncode)

In [ ]:
import numpy as np, pathlib
files = sorted(pathlib.Path('data/landmarks').rglob('*.npy'))
print('npy files:', len(files))
for f in files[:5]:
    a = np.load(f); print(f, a.shape, a.dtype, 'NaN', int(np.isnan(a).sum()))

# 5) Full extraction

Set `INCLUDE_FACE` below. **True -> D=1632 (face on)** is the default and matches local `include_face: true`. **False -> D=234 (`--no_face`)** is ~3x faster/smaller. Whichever you pick MUST match `training/config.yaml` `data.include_face` locally. The job is resumable (skips existing `.npy`). If Kaggle times out, just re-run this cell.

In [ ]:
import subprocess, sys
INCLUDE_FACE = True   # True -> D=1632 ; False -> D=234 (faster)
cmd = [sys.executable, 'preprocessing/extract_landmarks.py',
       '--raw_dir', _RAW_DIR, '--out_dir', 'data/landmarks']
if not INCLUDE_FACE:
    cmd.append('--no_face')
r = subprocess.run(cmd)
print('exit', r.returncode)

In [ ]:
import pathlib, shutil
src = pathlib.Path('data/landmarks')
files = list(src.rglob('*.npy'))
print('total npy:', len(files))
print('classes (word folders):', len({f.parent.name for f in files if f.parent.name.lower()!='extra'}))
shutil.make_archive('landmarks', 'zip', src)
z = pathlib.Path('landmarks.zip')
print('zip ready:', z, round(z.stat().st_size/1e6,1), 'MB -> download from output panel')

## Download & train locally
1. Download `landmarks.zip` from the output panel (File icon, top-right).
2. Unzip into the project: `Expand-Archive landmarks.zip -DestinationPath data/landmarks`.
3. Ensure `data.include_face` in `training/config.yaml` equals `INCLUDE_FACE` above.
4. `python utils/make_splits.py --top_n 50`
5. `python training/train.py` (on your RTX 5050)
6. `python training/evaluate.py` + `python inference/live_predict.py`
